In [ ]:
# web 문서를 읽어 RAG 구현
# 웹에서 텍스트 추출 -> 임베딩 -> ChromaDB에 저장 -> 질문과 유사한 문서 검색 -> LLM요청 후 결과 생성

!pip install openai sentence-transformers chromadb python-dotenv
!pip install requests beautifulsoup4

In [ ]:
from openai import OpenAI
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
from chromadb import PersistentClient
import os
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
model = "gpt-4o-mini"
embedder = SentenceTransformer("all-MiniLM-L6-v2")


In [ ]:
from huggingface_hub import delete_collection
# 웹페이지에서 텍스트 읽기
def extract_from_urlFunc(url):
    headers = {'User-Agent':'Mozilla/5.0'}
    resp = requests.get(url, headers=headers)
    # print('status_code', resp.status_code)

    if resp.status_code != 200:
        print('요청 실패, html preview', resp.text[:200])
        return []

    soup = BeautifulSoup(resp.text, 'html.parser')
    paragraphs = soup.find_all("p")

    texts = [  # p 태그 안의 텍스트만 빼내고 빈 문자열은 제외
        p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)
    ]
    print('found p count : ', len(texts))
    return texts


# ChromaDB에 저장
chroma_client = PersistentClient(path="./chroma_db")

try:
    chroma_client.delete_collection("webdata")
except:
    pass

collection = chroma_client.create_collection(
    name = "webdata",
    metadata={"hnsw:space":"cosine"}
)

url = "https://ko.wikipedia.org/wiki/김치찌게"
web_docs = extract_from_urlFunc(url)
print(web_docs)

if not web_docs:
    print('추출된 문서가 없어요')
    raise Exception('문서 없음')

# web_docs를 벡터로 변환
web_embeddings = embedder.encode(web_docs, normalize_embeddings=True)
print('web_embeddings : ', web_embeddings.shape)

# vectordb에 저장
for i, (doc, emb) in enumerate(zip(web_docs, web_embeddings), start=0):
    collection.add(
        documents = [doc],
        embeddings = [emb.tolist()],
        ids = [f'web_{i}']
    )

print('저장된 문서 수 : ', collection.count())

result = collection.get(ids=['web_0', 'web_1'])
print('샘플 문서 : ', result['documents'])

In [ ]:
# LLM에게 질문(prompt)하기 위한 질문 내용 강화
query = "김치찌게의 역사와 조리법 알려 줘"

query_vec = embedder.encode(
    [query], normalize_embeddings=True
)[0]

# print(query_vec)

# 질문벡터와 유사한 문서 검색
results = collection.query(
    query_embeddings=[query_vec.tolist()],
    n_results=3,
    include=['documents', 'distances']
)

retrieved_docs = results['documents'][0]
retrieved_dist = results['distances'][0]

for i, (doc, dist) in enumerate(zip(retrieved_docs, retrieved_dist), 1):
    print(f'문서{i} : {doc}')
    print(f'distances : {dist:.4f}\n')


# 검색 기반 프롬프트 구성
retrieved_text = '\n'.join(retrieved_docs)
print(retrieved_text)

In [ ]:
prompt = f"""
너는 한국요리 특히 찌게 전문가야.
다음 문서를 참고하여 '{query}'에 대해 답변해줘

[참고문서]
{retrieved_text}

요구 사항:
김치찌게의 조리법, 재료의 역할까지 설명해.
최소 20문장 이내로 작성해.
마크다운 없이 작성해.
각 문장을 줄바꿈으로 분리해.
"""

# print('prompt 내용 : ', prompt)

# LLM에게 질문 후 결과 얻기
response = client.responses.create(
    model=model,
    input=prompt
)
print('답변 : ', response.output_text)